### Problem statement

You have $N$ positive datapoints. Each datapoint $i$ has:
- a document $d_i$ with length $L_i$
- a single evidence sentence $e_i$

We want to create ⌊N/2⌋ new datapoints. Each new datapoint is formed by combining two original datapoints (so it contains two evidence sentences). The objective we care about: keep the maximum new datapoint length small -- since the text being too long might exceed the context length of our LLMs.

### Simple optimal-greedy algorithm for minimizing maximum pair-sum

- Sort datapoints by length.
- Pair the longest remaining item with the shortest remaining item.

Why: pairing largest with smallest minimizes the largest pair sum (standard proof by exchange argument). Complexity: $O(N log N)$ for sort, $O(N)$ for pairing. Works best when each pair must contain exactly two items.

In [2]:
import json

with open('ContraDoc.json', 'r') as f:
    data = json.load(f)

print(len(data['pos'])) # number of positive examples = 449
print(len(data['neg'])) # number of negative examples = 442

449
442


In [ ]:
pos_data = data['pos']

count = 0
for doc_id in pos_data.keys():
    print('-> count:', count)
    text = pos_data[doc_id]['text']
    print('-> text:\n', text)
    evidence = pos_data[doc_id]['evidence']
    print('-> evidence:\n', evidence)
    print('-> evidence length:\n', len(evidence))

### Creating the positive dataset

In [ ]:
import csv

# --- Load positive data ---
pos_data = data['pos']

# --- Collect items with length ---
items = []
for doc_id in pos_data.keys():
    text = pos_data[doc_id]['text']
    evidence = pos_data[doc_id]['evidence']
    
    length = len(text)
    
    items.append({
        'doc_id': doc_id,
        'text': text,
        'evidence': evidence,
        'length': length
    })

# --- Sort by length (ascending) ---
items.sort(key=lambda x: x['length'])

# --- Pair shortest with longest ---
pairs = []
i = 0
j = len(items) - 1

while i < j:
    pairs.append((items[i], items[j]))
    i += 1
    j -= 1

# Optional: handle leftover if odd number of datapoints
if i == j:
    print(f"Warning: one leftover datapoint (doc_id={items[i]['doc_id']}) not paired.")

# --- Write to CSV ---
output_file = "positive_data.csv"

with open(output_file, mode='w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f, delimiter=';')
    
    # Header
    writer.writerow(['text', 'evidence 1', 'evidence 2'])
    
    # Rows
    for item1, item2 in pairs:
        combined_text = item1['text'] + "\n" + item2['text']
        evidence1 = item1['evidence']
        evidence2 = item2['evidence']
        
        writer.writerow([combined_text, evidence1, evidence2])

print(f"Done. Wrote {len(pairs)} paired datapoints to {output_file}") # expect floor(449 / 2) = 224 pairs

Done. Wrote 224 paired datapoints to paired_contradoc_pos.csv


### Creating the negative dataset

In [ ]:
import csv

# --- Load negative data ---
neg_data = data['neg']

# --- Collect items with length ---
items = []
for doc_id in neg_data.keys():
    text = neg_data[doc_id]['text']
    
    items.append({
        'doc_id': doc_id,
        'text': text,
    })

# --- Write to CSV ---
output_file = "negative_data.csv"

with open(output_file, mode='w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f, delimiter=';')
    
    # Header
    writer.writerow(['text',])
    
    # Rows
    for item in items:
        writer.writerow([item['text']])

print(f"Done")

Done
